# 02 — Coding the Wikipedia indicator

This is the hard part of the project, and the part that normally gets done by
hand. Assigning a Wikipedia article to an IPO firm involves several judgment
calls, plus a couple of exclusions.

### The coding rule

An IPO firm is coded `Wikipedia = 1` when an article about it existed **before
the first trading day** with **at least 30 words in the main body**. Redirect
pages and disambiguation-style mentions are coded 0.

### What we automate

Matching on the firm's own name, plus **redirect following**, which catches
renames for free (`Liberty Oilfield Services` → `Liberty Energy`). The harder
relationships — parent company, major subsidiary, spun-off entity,
predecessor, core product — are judgment calls and are *not* automated.

### Three-way outcome, not two

A naive matcher silently guesses on hard cases and biases the result. Ours
returns `accept`, `reject`, or **`ambiguous`** — the last routed to human review
with the evidence attached. The motivating pair:

- `Nu Holdings` → `Nubank` — zero token overlap, and **correct**
- `Array Technologies` → `ATI Technologies` — 0.5 overlap, and **wrong**
  (ATI was formerly "Array Technology Inc"; different company entirely)

Nothing in the API distinguishes these, so neither is auto-accepted.

In [1]:
import json, collections
import pipeline as P

rows = json.loads((P.OUT / "panel.json").read_text())
print(f"panel rows: {len(rows)}")
print("wiki status:", collections.Counter(r["wiki_status"] for r in rows))

panel rows: 899
wiki status: Counter({'reject': 533, 'accept': 235, 'ambiguous': 131})


## Validation against known answers

A small set of firms whose correct coding can be established independently
serves as a regression test. A case passes if it is either coded correctly
automatically, **or** correctly escalated to review (i.e. a reviewer looking at
the proposed title would reach the right answer).

In [2]:
auto = esc = wrong = 0
print(f"{'firm':32s} {'exp':>3s}  outcome")
print("-" * 78)
for name, date, exp, note in P.BENCHMARK_CASES:
    r = P.wikipedia_flag(name, date)
    if r["status"] == "accept":
        good = r["wiki"] == exp
        auto += good; wrong += not good
        tag = f"AUTO wiki={r['wiki']}" + ("" if good else "  <-- WRONG")
    else:
        wb = r.get("would_be")
        good = wb == exp
        esc += good; wrong += not good
        tag = f"REVIEW would_be={wb}" + ("" if good else "  <-- MISLEADING")
    print(f"{name:32s} {exp:>3d}  {tag:34s} {str(r['title'])[:24]}")
print("-" * 78)
print(f"auto-correct {auto} | correctly escalated {esc} | wrong {wrong}  (of {len(P.BENCHMARK_CASES)})")

firm                             exp  outcome
------------------------------------------------------------------------------


LinkedIn                           1  AUTO wiki=1                        LinkedIn


Allot Communications               0  REVIEW would_be=0                  Allot Ltd.


Zoetis                             1  AUTO wiki=1                        Zoetis


Veridian                           0  AUTO wiki=0                        Veridian


The Hertz Corporation              1  REVIEW would_be=1                  Hertz Global Holdings


New York Mercantile Exchange       1  AUTO wiki=1                        New York Mercantile Exch
------------------------------------------------------------------------------
auto-correct 4 | correctly escalated 2 | wrong 0  (of 6)


## The human-review queue

Ambiguous cases are written to `data/out/review_queue.csv` with the proposed
article, its creation date, body word count and lead extract, so a reviewer can
adjudicate quickly. Notebook 03 reports the result three ways — excluding these
rows, coding them all 1, and coding them all 0 — so the conclusion can be
checked against the worst case.

In [3]:
import csv
amb = [r for r in rows if r["wiki_status"] == "ambiguous"]
print(f"ambiguous cases needing review: {len(amb)}")

out = P.OUT / "review_queue.csv"
with open(out, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["ticker", "company", "ipo_date", "proposed_article",
                "article_created", "body_words_at_ipo", "overlap",
                "auto_would_be", "lead_extract", "VERDICT_1_or_0"])
    for r in sorted(amb, key=lambda x: -(x["wiki_overlap"] or 0)):
        w.writerow([r["ticker"], r["name"], r["ipo_date"], r["wiki_title"],
                    (r["wiki_created"] or "")[:10], r["wiki_words"],
                    r["wiki_overlap"], r["wiki_would_be"],
                    (r["wiki_extract"] or "")[:200], ""])
print("wrote", out)

for r in sorted(amb, key=lambda x: -(x["wiki_overlap"] or 0))[:15]:
    print(f"  {r['ticker']:6s} {r['name'][:26]:26s} -> {str(r['wiki_title'])[:26]:26s} "
          f"ov={r['wiki_overlap']} would_be={r['wiki_would_be']}")

ambiguous cases needing review: 131
wrote /Users/daniel/Documents/coding_stuff/IPOTracker/research/data/underpricing/out/review_queue.csv
  WVE    WaVe Life Sciences Ltd     -> BIT Life Sciences          ov=0.667 would_be=1
  TPB    Turning Point Brands Inc   -> TNA Turning Point (2025)   ov=0.667 would_be=0
  FHB    First Hawaiian             -> First Hawaiian Bank        ov=0.667 would_be=1
  CBNK   Capital Bancorp Inc        -> Pacific Capital Bancorp    ov=0.667 would_be=1
  KRUS   Kura Sushi USA Inc         -> Kura Sushi                 ov=0.667 would_be=1
  FSBC   Five Star Bancorp          -> Five Star Bank (California ov=0.667 would_be=1
  FA     First Advantage Corp       -> First-mover advantage      ov=0.667 would_be=1
  BROS   Dutch Bros Inc             -> Dutch Bros Coffee          ov=0.667 would_be=1
  COCO   The Vita Coco Company, Inc -> Vita Coco                  ov=0.667 would_be=0
  EDBL   Edible Garden AG Inc       -> St. Raphael's Edible Garde ov=0.667 would_be=1
  

## Auto-adjudicating the grey zone

Two free signals resolve most ambiguous cases, both suggested by the
motivating pair:

1. **Industry alignment** — EDGAR gives each filer's SIC description, compared
   against the article's lead extract by substring containment (so
   "Oil & Gas Field Services" matches "oilfield services").
2. **Already defunct** — an entity acquired or dissolved well before the IPO
   date cannot be the company going public. This is what separates
   `ATI Technologies` (bought by AMD in 2006) from the 2020 solar-tracker IPO.

Whatever these cannot settle stays in the queue for a human.

In [4]:
adj = []
for r in amb:
    v, why, sig = P.refine_ambiguous(r)
    adj.append((r, v, why, sig))

auto = [x for x in adj if x[1] is not None]
human = [x for x in adj if x[1] is None]
print(f"ambiguous          : {len(adj)}")
print(f"  auto-adjudicated : {len(auto)}")
print(f"  still for human  : {len(human)}")

print("\nvalidation on the confirmed cases:")
for c in P.ADJUDICATED_CASES:
    hit = [x for x in adj if x[0]["name"].lower().startswith(c["name"].lower()[:12])]
    if hit:
        r, v, why, sig = hit[0]
        print(f"  {c['name']:22s} truth={c['truth']} verdict={v} [{'OK' if v==c['truth'] else 'CHECK'}] {why}")
    else:
        print(f"  {c['name']:22s} (not in this sample)")

ambiguous          : 131
  auto-adjudicated : 103
  still for human  : 28

validation on the confirmed cases:
  Nu Holdings            truth=1 verdict=1 [OK] industry matches the filer's SIC
  Array Technologies     truth=0 verdict=None [CHECK] needs human review


In [5]:
# Apply the auto-adjudications back onto the panel rows.
by_key = {(r["ticker"], r["ipo_date"]): r for r in rows}
for r, v, why, sig in auto:
    tgt = by_key[(r["ticker"], r["ipo_date"])]
    tgt["wiki_refined"] = v
    tgt["wiki_refined_why"] = why
for r, v, why, sig in human:
    by_key[(r["ticker"], r["ipo_date"])]["wiki_refined"] = None
for r in rows:
    if r["wiki_status"] == "accept":
        r["wiki_refined"] = r["wiki"]
    elif r["wiki_status"] == "reject":
        # No plausible company article found -> a real zero.
        r["wiki_refined"] = 0

(P.OUT / "panel_refined.json").write_text(json.dumps(rows, indent=1))
n_final = sum(1 for r in rows if r.get("wiki_refined") is not None)
print(f"rows with a usable indicator: {n_final}/{len(rows)}")
print("  of which Wikipedia=1:",
      sum(1 for r in rows if r.get("wiki_refined") == 1))

# Anything left genuinely needs eyes.
with open(P.OUT / "review_queue.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["ticker", "company", "ipo_date", "sic_desc", "proposed_article",
                "article_created", "body_words_at_ipo", "overlap",
                "lead_extract", "VERDICT_1_or_0"])
    for r, v, why, sig in human:
        w.writerow([r["ticker"], r["name"], r["ipo_date"], r.get("sic_desc"),
                    r["wiki_title"], (r["wiki_created"] or "")[:10],
                    r["wiki_words"], r["wiki_overlap"],
                    (r["wiki_extract"] or "")[:200], ""])
print(f"\nwrote review_queue.csv with {len(human)} rows needing a human")

# Human verdicts, once returned, take precedence over everything automated.
vpath = P.OUT / "review_verdicts.csv"
if vpath.exists():
    verdicts = {r["ticker"]: int(r["VERDICT"])
                for r in csv.DictReader(open(vpath)) if r.get("VERDICT") not in (None, "")}
    applied = 0
    for r in rows:
        if r["ticker"] in verdicts:
            r["wiki_refined"] = verdicts[r["ticker"]]
            r["wiki_verdict_source"] = "human"
            applied += 1
    (P.OUT / "panel_refined.json").write_text(json.dumps(rows, indent=1))
    print(f"applied {applied} human verdicts from review_verdicts.csv")
    print("  coded 1:", [t for t, v in verdicts.items() if v == 1])

rows with a usable indicator: 865/899
  of which Wikipedia=1: 210

wrote review_queue.csv with 28 rows needing a human
applied 28 human verdicts from review_verdicts.csv
  coded 1: ['SSTI', 'ELAN', 'ULS']


## Distribution of the indicator

In [6]:
acc = [r for r in rows if r["wiki_status"] == "accept"]
n1 = sum(1 for r in acc if r["wiki"] == 1)
print(f"accepted rows      : {len(acc)}")
print(f"  Wikipedia = 1    : {n1} ({n1/max(len(acc),1):.1%})")
print(f"  Wikipedia = 0    : {len(acc)-n1}")
print()
print("\nreasons for a zero:")
for reason, c in collections.Counter(
        r["wiki_reason"] for r in acc if r["wiki"] == 0).most_common():
    print(f"  {reason:24s} {c}")

accepted rows      : 235
  Wikipedia = 1    : 135 (57.4%)
  Wikipedia = 0    : 100


reasons for a zero:
  created_after_ipo        59
  under_30_words           32
  redirect_stub            3
